In [1]:
import os
!pip install einops


if os.path.isdir('Restormer'):
  !rm -r Restormer

# Clone Restormer
!git clone https://github.com/swz30/Restormer.git
%cd Restormer


d:\Automatic ANPR\Self_Code\anpr_dashboard\core\Restormer


Cloning into 'Restormer'...


In [2]:
!wget https://github.com/swz30/Restormer/releases/download/v1.0/deraining.pth \
-P core/Restormer/Deraining/pretrained_models


'wget' is not recognized as an internal or external command,
operable program or batch file.


In [4]:
import os
import urllib.request

url = "https://github.com/swz30/Restormer/releases/download/v1.0/deraining.pth"
save_path = r"D:\Automatic ANPR\Self_Code\anpr_dashboard\core\Restormer\Deraining\pretrained_models\deraining.pth"

os.makedirs(os.path.dirname(save_path), exist_ok=True)
urllib.request.urlretrieve(url, save_path)

print("Downloaded:", save_path)

Downloaded: D:\Automatic ANPR\Self_Code\anpr_dashboard\core\Restormer\Deraining\pretrained_models\deraining.pth


In [5]:
!git clone https://github.com/FVL2020/ICCV-2023-MB-TaylorFormer.git


Cloning into 'ICCV-2023-MB-TaylorFormer'...


In [6]:
pip install lmdb

Note: you may need to restart the kernel to use updated packages.


In [7]:
pip install torchstat

Note: you may need to restart the kernel to use updated packages.


In [1]:
!git clone https://github.com/megvii-research/NAFNet

Cloning into 'NAFNet'...


In [3]:
import torch
import cv2
import numpy as np
import sys
sys.path.append('.')

from RRDBNet_arch import RRDBNet

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Load model once
model = RRDBNet(3, 3, 64, 23, gc=32)
model.load_state_dict(torch.load(r'D:\Automatic ANPR\Self_Code\anpr_dashboard\core\ESRGAN\models\RRDB_ESRGAN_x4.pth'), strict=True)
model.eval()
model = model.to(device)


def esrgan_super_resolve_cv(img_bgr):
    """
    img_bgr: OpenCV image (numpy array, BGR)
    Returns: Super-resolved image (numpy array, BGR)
    """

    img = img_bgr.astype(np.float32) / 255.0

    # BGR → RGB and HWC → CHW
    img = torch.from_numpy(np.transpose(img[:, :, [2,1,0]], (2,0,1))).float()
    img = img.unsqueeze(0).to(device)

    with torch.no_grad():
        output = model(img).data.squeeze().float().cpu().clamp_(0, 1).numpy()

    # CHW → HWC and RGB → BGR
    output = np.transpose(output[[2,1,0], :, :], (1,2,0))
    output = (output * 255.0).round().astype(np.uint8)

    return output

In [6]:
img = cv2.imread(r"D:\Automatic ANPR\Self_Code\anpr_dashboard\debug_plate_crops\plate_crop44.jpg")   # cv2 image
sr_img = esrgan_super_resolve_cv(img)

cv2.imwrite("results/output.png", sr_img)

True

In [9]:
from skimage import img_as_ubyte
from runpy import run_path
import os

In [10]:
task = 'Single_Image_Defocus_Deblurring'

def get_weights_and_parameters(task, parameters):
    if task == 'Motion_Deblurring':
        weights = os.path.join('Motion_Deblurring', 'pretrained_models', 'motion_deblurring.pth')
    elif task == 'Single_Image_Defocus_Deblurring':
        weights = os.path.join('core', 'Restormer','Defocus_Deblurring', 'pretrained_models', 'net_g_2000 (2).pth.zip')
    elif task == 'Deraining':
        weights = os.path.join('Deraining', 'pretrained_models', 'deraining.pth')
    elif task == 'Real_Denoising':
        weights = os.path.join('Denoising', 'pretrained_models', 'real_denoising.pth')
        parameters['LayerNorm_type'] =  'BiasFree'
    return weights, parameters


# Get model weights and parameters
parameters = {'inp_channels':3, 'out_channels':3, 'dim':12, 'num_blocks':[2,2,2,2], 'num_refinement_blocks':2, 'heads':[1,2,2,4], 'ffn_expansion_factor':2.66, 'bias':False, 'LayerNorm_type':'WithBias', 'dual_pixel_task':False}
weights, parameters = get_weights_and_parameters(task, parameters)

load_arch = run_path(os.path.join('core','Restormer', 'basicsr', 'models', 'archs', 'restormer_arch.py'))
model_deblur = load_arch['Restormer'](**parameters)
model_deblur.cuda()

checkpoint = torch.load(weights)
model_deblur.load_state_dict(checkpoint['params'])
model_deblur.eval()


def restore_image(img, model, img_multiple_of=8):
    """
    img: numpy array (BGR or RGB)
    returns: restored numpy image (same size)
    """

    model.eval()

    # If BGR → RGB
    if img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    with torch.no_grad():
        input_ = torch.from_numpy(img).float().div(255.).permute(2,0,1).unsqueeze(0).cuda()

        h, w = input_.shape[2], input_.shape[3]
        H = ((h + img_multiple_of) // img_multiple_of) * img_multiple_of
        W = ((w + img_multiple_of) // img_multiple_of) * img_multiple_of
        padh = H - h if h % img_multiple_of != 0 else 0
        padw = W - w if w % img_multiple_of != 0 else 0

        input_ = F.pad(input_, (0, padw, 0, padh), 'reflect')

        restored = model(input_)
        restored = torch.clamp(restored, 0, 1)

        restored = restored[:, :, :h, :w]
        restored = restored.permute(0,2,3,1).cpu().numpy()[0]
        restored = img_as_ubyte(restored)

        # Back to BGR for OpenCV pipeline
        restored = cv2.cvtColor(restored, cv2.COLOR_RGB2BGR)

    return restored

FileNotFoundError: [Errno 2] No such file or directory: 'd:\\Automatic ANPR\\Self_Code\\anpr_dashboard\\core\\ESRGAN\\core\\Restormer\\basicsr\\models\\archs\\restormer_arch.py'